# Task 03 · Predictive Analytics Using Historical Data

**Internship:** Thiranex Data Analytics Internship
**Task:** Build a predictive model to forecast future trends from historical data

---

## Objective

Forecast future daily revenue using the historical sales transactions from
**Task 01** (`sales_data.csv`, Jan–Dec 2023). Two modeling approaches are
built and compared side by side:

1. **Time-series model — SARIMA** (captures trend + weekly seasonality directly from the series)
2. **Regression model — Log-Linear Regression** (trend + day-of-week features)

Both are evaluated on a held-out test period and used to forecast revenue
30 days beyond the end of the historical data.


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

DATA_PATH = '../data/raw_sales_data.csv'
OUTPUT_DIR = '../outputs'


## 2. Load & Inspect Historical Data

Reusing the transaction-level dataset from Task 01 (Sales & Revenue Dashboard).
Each row is one order; we will aggregate this into a daily revenue time series.

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


In [ ]:
df.info()
print()
print("Nulls per column:")
print(df.isnull().sum())
print()
print("Date range:", df['Date'].min(), "to", df['Date'].max())


## 3. Data Cleaning & Preprocessing

Steps:
- Parse dates
- Aggregate transaction-level data into **daily total revenue**
- Reindex to a *complete* daily calendar (the raw data has some missing dates)
- Interpolate missing days linearly (reasonable for smooth daily revenue gaps)
- Log-transform revenue — daily revenue grows roughly **exponentially** across
  the year as order volume scales up, and modeling on the log scale stabilizes
  variance and makes the trend close to linear (important for both models below).

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])

# Aggregate to daily revenue
daily = df.groupby('Date')['Revenue'].sum().reset_index().sort_values('Date')

# Reindex to a full daily calendar and find gaps
full_range = pd.date_range(daily['Date'].min(), daily['Date'].max(), freq='D')
print(f"Calendar days in range: {len(full_range)} | Days with data: {len(daily)} | Missing: {len(full_range) - len(daily)}")

daily = daily.set_index('Date').reindex(full_range)
daily.index.name = 'Date'
daily['Revenue'] = daily['Revenue'].interpolate(method='linear')
daily = daily.reset_index()

# Log-transform (log1p handles any zero values safely)
daily['log_revenue'] = np.log1p(daily['Revenue'])

print("\nFinal daily series shape:", daily.shape)
daily.head()


## 4. Exploratory Data Analysis

Visualize the overall trend, growth pattern, and weekly seasonality before modeling.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9))

axes[0].plot(daily['Date'], daily['Revenue'], color='#2563eb', linewidth=1.2)
axes[0].set_title('Daily Revenue — Full Year 2023 (Raw Scale)')
axes[0].set_ylabel('Revenue')

axes[1].plot(daily['Date'], daily['log_revenue'], color='#16a34a', linewidth=1.2)
axes[1].set_title('Daily Revenue — Log Scale (note the near-linear trend)')
axes[1].set_ylabel('log(1 + Revenue)')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_revenue_trend.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
daily['day_of_week'] = daily['Date'].dt.day_name()
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

plt.figure(figsize=(10, 5))
sns.boxplot(data=daily, x='day_of_week', y='Revenue', order=dow_order, palette='Blues')
plt.title('Revenue Distribution by Day of Week')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_weekly_seasonality.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
daily['month'] = daily['Date'].dt.month_name()
monthly = daily.groupby(daily['Date'].dt.to_period('M'))['Revenue'].sum()

plt.figure(figsize=(11, 5))
monthly.plot(kind='bar', color='#7c3aed')
plt.title('Total Revenue by Month — 2023')
plt.ylabel('Revenue')
plt.xlabel('Month')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03_monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Train / Test Split

The last **30 days** of 2023 are held out as a test set to evaluate forecast
accuracy. Everything before that is used for training — this respects the
time-ordering of the data (no shuffling, as is standard for time series).

In [ ]:
TEST_DAYS = 30

train = daily.iloc[:-TEST_DAYS].copy()
test = daily.iloc[-TEST_DAYS:].copy()

print(f"Train: {train['Date'].min().date()} -> {train['Date'].max().date()}  ({len(train)} days)")
print(f"Test:  {test['Date'].min().date()} -> {test['Date'].max().date()}  ({len(test)} days)")


## 6. Model A — Time-Series Forecasting (SARIMA)

SARIMA (Seasonal AutoRegressive Integrated Moving Average) models the series
directly using its own past values, trend, and a 7-day seasonal cycle
(weekly pattern). Fit on log-revenue for the reasons discussed above.

`order=(2,1,2)` handles the non-seasonal autoregression/differencing/moving-average
terms; `seasonal_order=(1,1,1,7)` captures a weekly seasonal cycle.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

ts_train_log = train.set_index('Date')['log_revenue']

sarima_model = SARIMAX(
    ts_train_log,
    order=(2, 1, 2),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)
# maxiter raised + Powell optimizer used for more reliable convergence on this series
sarima_fit = sarima_model.fit(disp=False, maxiter=200, method='powell')

forecast_sarima_log = sarima_fit.forecast(steps=TEST_DAYS)
forecast_sarima = np.expm1(forecast_sarima_log)
forecast_sarima.index = test['Date'].values

print(sarima_fit.summary().tables[0])


## 7. Model B — Regression-Based Forecasting

A log-linear regression on:
- `t` — day index (captures the overall trend; on the log scale, exponential
  growth becomes linear, which is exactly what linear regression is built for)
- Day-of-week dummy variables (captures weekly seasonality)

This is a deliberately simple, interpretable model — a good contrast to SARIMA.
*(Note: tree-based models like Random Forest were tested first but cannot
extrapolate beyond the max value seen in training, which fails badly on a
strongly trending series — linear regression on the log scale handles this
correctly.)*

In [ ]:
from sklearn.linear_model import LinearRegression

def add_features(d):
    d = d.copy()
    d['day_of_week_num'] = d['Date'].dt.dayofweek
    d['t'] = (d['Date'] - daily['Date'].min()).dt.days
    dummies = pd.get_dummies(d['day_of_week_num'], prefix='dow', drop_first=True)
    return pd.concat([d, dummies], axis=1)

full_feat = add_features(daily)
train_feat = full_feat.iloc[:-TEST_DAYS]
test_feat = full_feat.iloc[-TEST_DAYS:]

dow_cols = [c for c in full_feat.columns if c.startswith('dow_')]
feature_cols = ['t'] + dow_cols

X_train, y_train = train_feat[feature_cols], train_feat['log_revenue']
X_test = test_feat[feature_cols]

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

pred_reg_log = lin_reg.predict(X_test)
forecast_reg = pd.Series(np.expm1(pred_reg_log), index=test['Date'].values)

print("Regression coefficients (log scale):")
for name, coef in zip(feature_cols, lin_reg.coef_):
    print(f"  {name:10s} {coef:+.5f}")


## 8. Model Evaluation

Comparing both models on the held-out 30-day test period using:
- **MAE** (Mean Absolute Error)
- **RMSE** (Root Mean Squared Error)
- **MAPE** (Mean Absolute Percentage Error) — most interpretable for business reporting

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

y_true = test['Revenue'].values

results = pd.DataFrame({
    'Model': ['SARIMA (time-series)', 'Log-Linear Regression'],
    'MAE': [
        mean_absolute_error(y_true, forecast_sarima.values),
        mean_absolute_error(y_true, forecast_reg.values)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_true, forecast_sarima.values)),
        np.sqrt(mean_squared_error(y_true, forecast_reg.values))
    ],
    'MAPE (%)': [
        mape(y_true, forecast_sarima.values),
        mape(y_true, forecast_reg.values)
    ]
})

results_display = results.copy()
results_display['MAE'] = results_display['MAE'].map('{:,.0f}'.format)
results_display['RMSE'] = results_display['RMSE'].map('{:,.0f}'.format)
results_display['MAPE (%)'] = results_display['MAPE (%)'].map('{:.2f}'.format)
results_display


**Interpretation:** the model with the lower MAPE generalizes better on
this test window. Daily revenue here is noisy and the test period sits in the
steepest, most volatile part of the year's growth curve, so double-digit MAPE
is expected for *daily* granularity — aggregating forecasts to weekly/monthly
totals (shown later) reduces this error substantially, which is typical in
real-world forecasting.

## 9. Visualize Predictions vs. Actuals (Side by Side)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

# Show last 60 days of train for context + test period
context = daily.iloc[-(TEST_DAYS + 60):-TEST_DAYS]
ax.plot(context['Date'], context['Revenue'], label='Historical (train)', color='#94a3b8')
ax.plot(test['Date'], y_true, label='Actual', color='black', linewidth=2)
ax.plot(test['Date'], forecast_sarima.values, label='SARIMA forecast', color='#2563eb', linestyle='--', marker='o', markersize=3)
ax.plot(test['Date'], forecast_reg.values, label='Log-Linear Regression forecast', color='#dc2626', linestyle='--', marker='s', markersize=3)

ax.axvline(test['Date'].iloc[0], color='gray', linestyle=':', alpha=0.7)
ax.set_title('Actual vs. Predicted Daily Revenue — Test Period (Last 30 Days)')
ax.set_ylabel('Revenue')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/04_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)

axes[0].scatter(y_true, forecast_sarima.values, alpha=0.7, color='#2563eb')
lims = [min(y_true.min(), forecast_sarima.min()), max(y_true.max(), forecast_sarima.max())]
axes[0].plot(lims, lims, 'k--', alpha=0.5, label='Perfect prediction')
axes[0].set_title('SARIMA: Predicted vs Actual')
axes[0].set_xlabel('Actual Revenue')
axes[0].set_ylabel('Predicted Revenue')
axes[0].legend()

axes[1].scatter(y_true, forecast_reg.values, alpha=0.7, color='#dc2626')
lims2 = [min(y_true.min(), forecast_reg.min()), max(y_true.max(), forecast_reg.max())]
axes[1].plot(lims2, lims2, 'k--', alpha=0.5, label='Perfect prediction')
axes[1].set_title('Log-Linear Regression: Predicted vs Actual')
axes[1].set_xlabel('Actual Revenue')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/05_prediction_scatter.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Forecasting Future Trends (Next 30 Days Beyond Available Data)

Now refit both models on the **full** historical dataset (train + test) and
project 30 days into the future, beyond 2023-12-28.

In [ ]:
FUTURE_DAYS = 30

# --- SARIMA refit on full data, forecast forward ---
ts_full_log = daily.set_index('Date')['log_revenue']
sarima_full = SARIMAX(
    ts_full_log, order=(2, 1, 2), seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False, enforce_invertibility=False
).fit(disp=False, maxiter=200, method='powell')

future_dates = pd.date_range(daily['Date'].max() + pd.Timedelta(days=1), periods=FUTURE_DAYS, freq='D')
future_log_sarima = sarima_full.forecast(steps=FUTURE_DAYS)
future_sarima = pd.Series(np.expm1(future_log_sarima.values), index=future_dates)

# --- Log-linear regression refit on full data, forecast forward ---
full_feat_all = add_features(daily)
dow_cols_all = [c for c in full_feat_all.columns if c.startswith('dow_')]
lin_full = LinearRegression().fit(full_feat_all[['t'] + dow_cols_all], full_feat_all['log_revenue'])

future_df = pd.DataFrame({'Date': future_dates})
future_df['day_of_week_num'] = future_df['Date'].dt.dayofweek
future_df['t'] = (future_df['Date'] - daily['Date'].min()).dt.days
future_dummies = pd.get_dummies(future_df['day_of_week_num'], prefix='dow', drop_first=True)
future_df = pd.concat([future_df, future_dummies], axis=1)
for col in dow_cols_all:
    if col not in future_df.columns:
        future_df[col] = 0

future_pred_log = lin_full.predict(future_df[['t'] + dow_cols_all])
future_reg = pd.Series(np.expm1(future_pred_log), index=future_dates)

forecast_table = pd.DataFrame({
    'Date': future_dates,
    'SARIMA_Forecast': future_sarima.values,
    'LogLinearRegression_Forecast': future_reg.values
})
forecast_table.to_csv(f'{OUTPUT_DIR}/future_30_day_forecast.csv', index=False)
forecast_table.head(10)


In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

recent = daily.iloc[-90:]
ax.plot(recent['Date'], recent['Revenue'], label='Historical', color='#1f2937')
ax.plot(future_dates, future_sarima.values, label='SARIMA forecast (next 30 days)', color='#2563eb', linestyle='--')
ax.plot(future_dates, future_reg.values, label='Log-Linear Regression forecast (next 30 days)', color='#dc2626', linestyle='--')
ax.axvline(daily['Date'].max(), color='gray', linestyle=':', alpha=0.7, label='End of historical data')

ax.set_title('30-Day Future Revenue Forecast')
ax.set_ylabel('Revenue')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/06_future_forecast.png', dpi=150, bbox_inches='tight')
plt.show()


## 11. Summary & Key Learnings

- **Best performing model on held-out test data:** see the evaluation table in
  Section 8 — whichever model had the lower MAPE generalizes better for this
  series.
- **SARIMA** captures autocorrelation and weekly seasonality directly from the
  series, which helps with short-term local fluctuations.
- **Log-linear regression** captures the dominant exponential growth trend
  very cleanly once log-transformed, and is simple, fast, and interpretable.
- Both models agree on the **overall upward trend** for the next 30 days,
  which is the most actionable takeaway for business planning even if exact
  daily values differ.

### Key Learnings
- Predictive modeling with both time-series (SARIMA) and regression approaches
- Importance of log-transforming strongly trending/exponential data before modeling
- Data preprocessing: handling missing dates in a time series via interpolation
- Train/test splitting that respects chronological order (no shuffling)
- Model evaluation using MAE, RMSE, and MAPE
- Visualizing forecasts vs. actuals to communicate model performance
- Forecasting beyond the available historical range
